In [1]:
!python main.py

python: can't open file 'C:\\Users\\Lenovo\\main.py': [Errno 2] No such file or directory


In [2]:
pip install pygame

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pygame
import random
import sys

# Khởi tạo
pygame.init()

# Hằng số 
SCREEN_W, SCREEN_H = 400, 600
FPS            = 60
GRAVITY        = 0.45
JUMP_FORCE     = -8.5
PIPE_SPEED     = 3
PIPE_GAP       = 160       
PIPE_INTERVAL  = 1600      
GROUND_H       = 80        

# Màu sắc 
SKY        = (113, 197, 207)
WHITE      = (255, 255, 255)
BLACK      = (  0,   0,   0)
YELLOW     = (255, 204,   0)
YELLOW_LT  = (255, 230, 120)
BLUE_AI    = ( 50, 150, 255)
BLUE_LT    = (150, 220, 255)
ORANGE     = (255, 130,   0)
GREEN      = ( 78, 154,   6)
GROUND_CLR = (222, 184, 135)
DIRT       = (139,  90,  43)
GRASS      = (100, 160,  30)

class Bird:
    WING_FRAMES = 3
    WING_DELAY  = 100

    def __init__(self, x: int, y: int, color=YELLOW, light_color=YELLOW_LT):
        self.start_x, self.start_y = x, y
        self.color = color
        self.light_color = light_color
        self.reset()

    def reset(self):
        self.x = float(self.start_x)
        self.y = float(self.start_y)
        self.velocity = 0.0
        self.rotation = 0.0
        self.wing_frame = 0
        self.wing_timer = 0
        self.alive = True

    def jump(self):
        if self.alive:
            self.velocity = JUMP_FORCE

    def update(self, dt: int):
        if not self.alive:
            self.x -= PIPE_SPEED 
            return
        
        self.velocity += GRAVITY
        self.velocity = min(self.velocity, 14)
        self.y += self.velocity
        self.rotation = max(-30, min(self.velocity * 4.5, 70))
        self.wing_timer += dt
        if self.wing_timer >= self.WING_DELAY:
            self.wing_timer = 0
            self.wing_frame = (self.wing_frame + 1) % self.WING_FRAMES

    def draw(self, screen: pygame.Surface):
        W, H = 44, 32
        surf = pygame.Surface((W + 22, H + 12), pygame.SRCALPHA)
        ox, oy = 10, 5
        alpha = 255 if self.alive else 120 

        pygame.draw.ellipse(surf, (*self.color, alpha), (ox, oy, W, H))
        pygame.draw.ellipse(surf, (*self.light_color, alpha), (ox + 5, oy + 11, 18, 14))
        wing_dy = [-7, 0, 7][self.wing_frame]
        pygame.draw.ellipse(surf, (*ORANGE, alpha), (ox + 3, oy + 10 + wing_dy, 16, 10))
        pygame.draw.ellipse(surf, (*WHITE, alpha), (ox + W - 14, oy + 4, 13, 13))
        pygame.draw.ellipse(surf, (*BLACK, alpha), (ox + W - 10, oy + 7, 7, 7))
        bx, by = ox + W - 2, oy + H // 2
        pygame.draw.polygon(surf, (*ORANGE, alpha), [(bx, by - 7), (bx + 13, by), (bx, by + 7)])

        rotated = pygame.transform.rotate(surf, -self.rotation)
        screen.blit(rotated, rotated.get_rect(center=(int(self.x), int(self.y))))

    def get_rect(self) -> pygame.Rect:
        return pygame.Rect(int(self.x) - 14, int(self.y) - 11, 28, 22)

class Pipe:
    WIDTH = 62
    CAP_H = 22
    def __init__(self, x: int, mode="normal"):
        self.x = float(x)
        self.mode = mode
        self.scored = False
        self.gap_y = random.randint(200, SCREEN_H - GROUND_H - 200)
        self.top = self.gap_y - PIPE_GAP // 2
        self.bottom = self.gap_y + PIPE_GAP // 2
        self.moved = False

    def update(self, bird_y=None, bird_vel=0):
        self.x -= PIPE_SPEED
        
        # Logic cột di chuyển (né chim)
        if self.mode == "moving_pipe" and bird_y is not None and not self.moved:
            if abs(self.x - 100) < 180:
                predicted_y = bird_y + bird_vel * 5
                if predicted_y > self.gap_y:
                    self.gap_y -= 100
                else:
                    self.gap_y += 100
                self.moved = True
                
                # Giới hạn biên cho gap_y
                min_y, max_y = 120, SCREEN_H - GROUND_H - 120
                self.gap_y = max(min_y, min(max_y, self.gap_y))
                self.top = self.gap_y - PIPE_GAP // 2
                self.bottom = self.gap_y + PIPE_GAP // 2

    def draw(self, screen: pygame.Surface):
        x_i = int(self.x)
        P_LIGHT, P_MID = (140, 200, 50), (78, 154, 6)
        def draw_part(y, h, is_top):
            if h <= 0: return
            r = pygame.Rect(x_i + 6, y, self.WIDTH - 12, h)
            pygame.draw.rect(screen, P_MID, r)
            pygame.draw.rect(screen, P_LIGHT, (x_i + 10, y, 6, h))
            pygame.draw.rect(screen, BLACK, r, 2)
            cy = (self.top - self.CAP_H) if is_top else self.bottom
            cr = pygame.Rect(x_i, cy, self.WIDTH, self.CAP_H)
            pygame.draw.rect(screen, P_MID, cr)
            pygame.draw.rect(screen, BLACK, cr, 2)
        
        draw_part(0, self.top - self.CAP_H, True)
        draw_part(self.bottom + self.CAP_H, SCREEN_H - self.bottom - self.CAP_H - GROUND_H, False)

class Game:
    def __init__(self):
        self.screen = pygame.display.set_mode((SCREEN_W, SCREEN_H))
        pygame.display.set_caption("Flappy Bird Ultimate")
        self.clock = pygame.time.Clock()
        self.mode = "normal" 
        self.show_mode_menu = False
        self.high_score = 0

        # Nút bấm
        self.btn_mode = pygame.Rect(SCREEN_W//2-100, SCREEN_H//2+100, 200, 45)
        self.btn_normal = pygame.Rect(SCREEN_W//2-100, SCREEN_H//2+155, 200, 40)
        self.btn_moving = pygame.Rect(SCREEN_W//2-100, SCREEN_H//2+200, 200, 40)
        self.btn_pvc = pygame.Rect(SCREEN_W//2-100, SCREEN_H//2+245, 200, 40)

        self.font_big = pygame.font.SysFont("Segoe UI", 52, bold=True)
        self.font_mid = pygame.font.SysFont("Segoe UI", 30, bold=True)
        self.font_small = pygame.font.SysFont("Segoe UI", 18, bold=True)
        
        self.bird = Bird(100, SCREEN_H//2, YELLOW, YELLOW_LT)
        self.ai_bird = Bird(100, SCREEN_H//2, BLUE_AI, BLUE_LT)
        
        self.clouds = [(55, 75), (195, 48), (325, 108), (140, 165), (290, 55)]
        self.reset()

    def reset(self):
        self.bird.reset()
        self.ai_bird.reset()
        self.pipes = []
        self.score = 0
        self.state = "waiting"
        self.pipe_timer = 0
        self.ground_x = 0
        self.result_text = ""

    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT: pygame.quit(); sys.exit()
            
            if event.type == pygame.MOUSEBUTTONDOWN:
                mx, my = event.pos
                if self.show_mode_menu:
                    if self.btn_normal.collidepoint(mx,my): self.mode = "normal"; self.show_mode_menu = False; self.reset()
                    elif self.btn_moving.collidepoint(mx,my): self.mode = "moving_pipe"; self.show_mode_menu = False; self.reset()
                    elif self.btn_pvc.collidepoint(mx,my): self.mode = "pvc"; self.show_mode_menu = False; self.reset()
                elif self.state == "waiting" and self.btn_mode.collidepoint(mx,my):
                    self.show_mode_menu = not self.show_mode_menu
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    if self.show_mode_menu: self.show_mode_menu = False
                    elif self.state in ["playing", "gameover"]:
                        self.state = "waiting"
                        self.reset()
                
                if event.key == pygame.K_SPACE:
                    if self.state == "waiting" and not self.show_mode_menu:
                        self.state = "playing"
                        self.bird.jump()
                    elif self.state == "playing":
                        self.bird.jump()
                    elif self.state == "gameover":
                        self.reset()

    def update(self, dt: int):
        if self.state != "playing": return

        # Update Chim
        self.bird.update(dt)
        if self.mode == "pvc":
            self.ai_logic()
            self.ai_bird.update(dt)

        # Sinh ống
        self.pipe_timer += dt
        if self.pipe_timer >= PIPE_INTERVAL:
            self.pipe_timer = 0
            self.pipes.append(Pipe(SCREEN_W + 20, self.mode))

        for p in self.pipes:
            # Truyền thông số chim để ống né (nếu ở chế độ moving_pipe)
            p.update(self.bird.y, self.bird.velocity)
            if not p.scored and p.x + p.WIDTH < self.bird.x:
                p.scored = True
                self.score += 1

        self.pipes = [p for p in self.pipes if p.x > -100]

        # Va chạm
        gy = SCREEN_H - GROUND_H
        targets = [self.bird]
        if self.mode == "pvc": targets.append(self.ai_bird)
        
        for b in targets:
            if not b.alive: continue
            if b.y + 15 >= gy or b.y - 15 <= 0: b.alive = False
            r = b.get_rect()
            for p in self.pipes:
                if pygame.Rect(p.x, 0, p.WIDTH, p.top).colliderect(r) or \
                   pygame.Rect(p.x, p.bottom, p.WIDTH, SCREEN_H).colliderect(r):
                    b.alive = False

        # Logic kết thúc game
        if not self.bird.alive:
            if self.mode == "pvc":
                if not self.ai_bird.alive: 
                    self.result_text = "Win"
                    self.state = "gameover"
                    self.high_score = max(self.high_score, self.score)
                else:
                    self.result_text = "AI Win"
                    self.state = "gameover"
                    self.high_score = max(self.high_score, self.score)
            else:
                self.result_text = "Game over"
                self.state = "gameover"
                self.high_score = max(self.high_score, self.score)
        elif self.mode == "pvc" and not self.ai_bird.alive:
             # AI chết trước nhưng người chơi vẫn bay
             pass 

    def ai_logic(self):
        if not self.ai_bird.alive: return
        target = next((p for p in self.pipes if p.x + p.WIDTH > self.ai_bird.x), None)
        if target:
            if self.ai_bird.y > target.gap_y + 10: self.ai_bird.jump()
        elif self.ai_bird.y > SCREEN_H//2: self.ai_bird.jump()

    def draw(self):
        self.screen.fill(SKY)
        for cx, cy in self.clouds:
            pygame.draw.ellipse(self.screen, WHITE, (cx - 32, cy, 64, 26))
            pygame.draw.ellipse(self.screen, WHITE, (cx - 18, cy - 16, 42, 28))
            pygame.draw.ellipse(self.screen, WHITE, (cx + 4,  cy -  2, 38, 22))
        
        for p in self.pipes: p.draw(self.screen)
        
        gy = SCREEN_H - GROUND_H
        if self.state == "playing": self.ground_x = (self.ground_x - PIPE_SPEED) % 40
        pygame.draw.rect(self.screen, GROUND_CLR, (0, gy, SCREEN_W, GROUND_H))
        pygame.draw.rect(self.screen, GRASS, (0, gy, SCREEN_W, 10))
        for gx in range(-40, SCREEN_W + 40, 40):
            pygame.draw.line(self.screen, DIRT, (gx+self.ground_x, gy+12), (gx+self.ground_x+15, gy+GROUND_H), 3)

        if self.mode == "pvc": self.ai_bird.draw(self.screen)
        self.bird.draw(self.screen)
        
        s_txt = str(self.score)
        sh = self.font_big.render(s_txt, True, BLACK)
        sc = self.font_big.render(s_txt, True, WHITE)
        self.screen.blit(sh, sh.get_rect(centerx=SCREEN_W//2+2, top=22))
        self.screen.blit(sc, sc.get_rect(centerx=SCREEN_W//2, top=20))

        if self.state == "waiting": self._draw_waiting()
        elif self.state == "gameover": self._draw_gameover()
        pygame.display.flip()

    def _draw_waiting(self):
        cx, cy = SCREEN_W//2, SCREEN_H//2
        sh = self.font_big.render("FLAPPY BIRD", True, BLACK)
        ti = self.font_big.render("FLAPPY BIRD", True, WHITE)
        self.screen.blit(sh, sh.get_rect(center=(cx+3, cy-120+3)))
        self.screen.blit(ti, ti.get_rect(center=(cx, cy-120)))

        def draw_btn(rect, text, color):
            pygame.draw.rect(self.screen, BLACK, (rect.x+2, rect.y+2, rect.width, rect.height), border_radius=8)
            pygame.draw.rect(self.screen, color, rect, border_radius=8)
            pygame.draw.rect(self.screen, WHITE, rect, 2, border_radius=8)
            s_sh = self.font_small.render(text, True, BLACK)
            s_tx = self.font_small.render(text, True, WHITE)
            self.screen.blit(s_sh, s_sh.get_rect(center=(rect.centerx+1, rect.centery+1)))
            self.screen.blit(s_tx, s_tx.get_rect(center=rect.center))

        m_name = "NGƯỜI VS MÁY" if self.mode == "pvc" else ("CỘT DI CHUYỂN" if self.mode == "moving_pipe" else "THƯỜNG")
        draw_btn(self.btn_mode, f"CHẾ ĐỘ: {m_name}", (50, 150, 250))

        if self.show_mode_menu:
            overlay = pygame.Surface((SCREEN_W, SCREEN_H), pygame.SRCALPHA)
            overlay.fill((0,0,0,80)); self.screen.blit(overlay, (0,0))
            draw_btn(self.btn_normal, "NORMAL", (80,80,80))
            draw_btn(self.btn_moving, "CỘT DI CHUYỂN", (200,50,50))
            draw_btn(self.btn_pvc, "NGƯỜI VS MÁY", (40, 180, 100))
        else:
            hint = self.font_small.render("Nhấn SPACE để bay", True, WHITE)
            self.screen.blit(hint, hint.get_rect(center=(cx, cy+50)))

    def _draw_gameover(self):
        overlay = pygame.Surface((SCREEN_W, SCREEN_H), pygame.SRCALPHA)
        overlay.fill((0,0,0,160)); self.screen.blit(overlay, (0,0))
        
        txt = self.result_text
        color = (100, 255, 100) if "Win" in txt else (255, 80, 80)
        
        sh = self.font_big.render(txt, True, BLACK)
        ti = self.font_big.render(txt, True, color)
        self.screen.blit(sh, sh.get_rect(center=(SCREEN_W//2+3, 220+3)))
        self.screen.blit(ti, ti.get_rect(center=(SCREEN_W//2, 220)))
        
        score = self.font_mid.render(f"Điểm: {self.score}", True, WHITE)
        self.screen.blit(score, score.get_rect(center=(SCREEN_W//2, 300)))
        high = self.font_small.render(f"Điểm cao nhất: {self.high_score}", True, (255, 255, 100))
        self.screen.blit(high, high.get_rect(center=(SCREEN_W//2, 330)))
        
        rst = self.font_small.render("Nhấn SPACE để chơi lại", True, WHITE)
        self.screen.blit(rst, rst.get_rect(center=(SCREEN_W//2, 360)))

    def run(self):
        while True:
            dt = self.clock.tick(FPS)
            self.handle_events()
            self.update(dt)
            self.draw()

if __name__ == "__main__":
    Game().run()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
